# 电化学反应的 EDBO+ 贝叶斯优化全流程

本 notebook 参考 `examples/tutorials/1_CLI_example.ipynb` 的 CLI 工作流，并使用 `experiment_conditions.md` 中给出的电化学条件空间。流程包括：

1. 定义组合条件空间并生成 CSV；
2. 在没有实验数据时让 EDBO+ 推荐初始实验；
3. 录入首轮实验结果；
4. 基于观测数据训练贝叶斯优化模型并推荐下一轮实验；
5. 查看全空间预测结果。

示例中的实验结果由一个虚拟响应函数生成，仅用于演示完整流程。真实实验时，把对应单元替换为实际测得的产率、选择性和电量/能耗即可。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 兼容从仓库根目录或 examples/electrochemistry 目录启动 Jupyter 的情况。
CWD = Path.cwd().resolve()
ROOT = next((path for path in [CWD, *CWD.parents] if (path / "edbo").exists()), CWD)
WORKDIR = ROOT / "examples" / "electrochemistry"
if not WORKDIR.exists():
    WORKDIR = CWD

sys.path.append(str(ROOT))

from edbo.plus.optimizer_botorch import EDBOplus

SCOPE_FILE = "electrochemistry_scope.csv"
ROUND0_FILE = "electrochemistry_round0.csv"
PRED_ROUND0_FILE = f"pred_{ROUND0_FILE}"

## 1. 定义电化学条件空间

`experiment_conditions.md` 给出的固定条件包括芴酮、苄甲醇、三苯膦和 10 mL 总溶剂量；可优化条件包括催化剂 C 组、电解质 E 组和溶剂配比 S 组。

下面的空间把固定条件也保留在 CSV 里，便于实验记录完整。电解质列表中原文件明确举例 4 种并注明“共 6 种”，这里补齐为同系列的 6 个候选；如果已有真实候选清单，直接替换 `electrolyte` 列表即可。

In [ ]:
reaction_components = {
    # 固体组分
    "fluorenone_mmol": [1.0],
    "catalyst": ["4-hydroxy-TEMPO", "TEMPO", "4-acetamido-TEMPO"],
    "catalyst_loading_mol_pct": [5, 10, 15],
    "electrolyte": [
        "n-Bu4NClO4",
        "n-Bu4NOAc",
        "n-Bu4NBF4",
        "Et4NClO4",
        "Et4NOAc",
        "Et4NBF4",
    ],
    "electrolyte_equiv": [0.05, 0.10, 0.20],
    "PPh3_equiv": [1.0],
    # 液体组分
    "benzyl_alcohol_equiv": [1.0, 1.5, 2.0],
    "solvent_ratio": [
        "MeCN:EtOAc=2:8",
        "MeCN:EtOAc=3:7",
        "MeCN:EtOAc=4:6",
        "MeCN:EtOAc=5:5",
        "MeCN:EtOAc=6:4",
        "MeCN:EtOAc=7:3",
    ],
    "solvent_total_mL": [10.0],
}

In [ ]:
scope = EDBOplus().generate_reaction_scope(
    components=reaction_components,
    directory=str(WORKDIR),
    filename=SCOPE_FILE,
    check_overwrite=False,
)

scope.head()

In [ ]:
df_scope = pd.read_csv(WORKDIR / SCOPE_FILE)
print(f"条件空间共有 {len(df_scope)} 个候选实验。")
df_scope.sample(5, random_state=0)

## 2. 没有观测数据时推荐初始实验

首次运行时，CSV 里还没有目标值。EDBO+ 会根据特征空间采样方法选择一批初始实验，并在 CSV 中加入目标列和 `priority` 列。

本示例优化 3 个目标：

- `yield_percent`：目标产物产率，越高越好；
- `selectivity_percent`：目标选择性，越高越好；
- `charge_F_per_mol`：单位底物通过电量，用作能耗/电化学成本代理，越低越好。

In [ ]:
OBJECTIVES = ["yield_percent", "selectivity_percent", "charge_F_per_mol"]
OBJECTIVE_MODE = ["max", "max", "min"]
BATCH_SIZE = 8

initial_suggestions = EDBOplus().run(
    directory=str(WORKDIR),
    filename=SCOPE_FILE,
    objectives=OBJECTIVES,
    objective_mode=OBJECTIVE_MODE,
    batch=BATCH_SIZE,
    columns_features="all",
    init_sampling_method="cvt",
    seed=0,
)

initial_suggestions.query("priority == 1")

## 3. 录入首轮实验结果

真实实验中，完成 `priority == 1` 的实验后，把每个目标的 `PENDING` 改为实测值即可。

为了让教程可以从头跑通，下面用一个虚拟响应函数模拟首轮实验结果。它只代表“如何把结果写回 CSV”，不代表真实反应规律。

In [ ]:
def virtual_electrochemistry_result(row, rng):
    """生成可复现的虚拟实验结果；真实项目中请替换为实测值。"""
    catalyst_bonus = {
        "4-hydroxy-TEMPO": 6,
        "TEMPO": 0,
        "4-acetamido-TEMPO": 10,
    }[row["catalyst"]]
    electrolyte_bonus = {
        "n-Bu4NClO4": 2,
        "n-Bu4NOAc": -2,
        "n-Bu4NBF4": 8,
        "Et4NClO4": 0,
        "Et4NOAc": -4,
        "Et4NBF4": 4,
    }[row["electrolyte"]]
    solvent_score = {
        "MeCN:EtOAc=2:8": -8,
        "MeCN:EtOAc=3:7": -3,
        "MeCN:EtOAc=4:6": 3,
        "MeCN:EtOAc=5:5": 8,
        "MeCN:EtOAc=6:4": 6,
        "MeCN:EtOAc=7:3": 0,
    }[row["solvent_ratio"]]

    loading_penalty = -1.4 * abs(row["catalyst_loading_mol_pct"] - 10)
    electrolyte_penalty = -45 * abs(row["electrolyte_equiv"] - 0.10)
    alcohol_penalty = -8 * abs(row["benzyl_alcohol_equiv"] - 1.5)

    yield_percent = 48 + catalyst_bonus + electrolyte_bonus + solvent_score
    yield_percent += loading_penalty + electrolyte_penalty + alcohol_penalty
    yield_percent += rng.normal(0, 2.0)
    yield_percent = float(np.clip(yield_percent, 0, 100))

    selectivity_percent = 62 + 0.45 * catalyst_bonus + 0.35 * electrolyte_bonus
    selectivity_percent += 0.6 * solvent_score - 3 * max(row["benzyl_alcohol_equiv"] - 1.5, 0)
    selectivity_percent += rng.normal(0, 1.5)
    selectivity_percent = float(np.clip(selectivity_percent, 0, 100))

    charge_F_per_mol = 2.6 - 0.015 * yield_percent + 0.25 * (row["electrolyte_equiv"] == 0.05)
    charge_F_per_mol += 0.18 * (row["solvent_ratio"] in ["MeCN:EtOAc=2:8", "MeCN:EtOAc=3:7"])
    charge_F_per_mol += rng.normal(0, 0.05)
    charge_F_per_mol = float(np.clip(charge_F_per_mol, 0.8, 3.5))

    return pd.Series(
        {
            "yield_percent": round(yield_percent, 1),
            "selectivity_percent": round(selectivity_percent, 1),
            "charge_F_per_mol": round(charge_F_per_mol, 2),
        }
    )

In [ ]:
df_round0 = pd.read_csv(WORKDIR / SCOPE_FILE)
initial_mask = df_round0["priority"] == 1
rng = np.random.default_rng(42)

simulated_results = df_round0.loc[initial_mask].apply(
    lambda row: virtual_electrochemistry_result(row, rng), axis=1
)
df_round0.loc[initial_mask, OBJECTIVES] = simulated_results[OBJECTIVES]

df_round0.to_csv(WORKDIR / ROUND0_FILE, index=False)
df_round0.loc[initial_mask, ["catalyst", "catalyst_loading_mol_pct", "electrolyte", "electrolyte_equiv", "benzyl_alcohol_equiv", "solvent_ratio", *OBJECTIVES]]

## 4. 用首轮观测数据推荐下一轮实验

现在 `electrochemistry_round0.csv` 中已经有一批非 `PENDING` 的观测值。再次运行 EDBO+ 时，它会训练代理模型，并为未测试条件分配新的优先级。

In [ ]:
next_suggestions = EDBOplus().run(
    directory=str(WORKDIR),
    filename=ROUND0_FILE,
    objectives=OBJECTIVES,
    objective_mode=OBJECTIVE_MODE,
    batch=BATCH_SIZE,
    columns_features="all",
    init_sampling_method="cvt",
    seed=1,
)

next_suggestions.query("priority == 1")

## 5. 查看全空间预测

当输入文件中包含观测值时，EDBO+ 会额外写出 `pred_<filename>`。该文件包含每个目标的预测均值、预测标准差和期望改进值，可用于理解模型为什么推荐某些实验。

In [ ]:
df_predictions = pd.read_csv(WORKDIR / PRED_ROUND0_FILE)

prediction_columns = [
    "priority",
    "yield_percent_predicted_mean",
    "yield_percent_predicted_std_dev",
    "yield_percent_expected_improvement",
    "selectivity_percent_predicted_mean",
    "selectivity_percent_predicted_std_dev",
    "selectivity_percent_expected_improvement",
    "charge_F_per_mol_predicted_mean",
    "charge_F_per_mol_predicted_std_dev",
    "charge_F_per_mol_expected_improvement",
]

df_predictions[
    ["catalyst", "catalyst_loading_mol_pct", "electrolyte", "electrolyte_equiv", "benzyl_alcohol_equiv", "solvent_ratio", *OBJECTIVES, *prediction_columns]
].head(12)

## 6. 进入下一轮

实际优化时，重复以下循环即可：

1. 按 `priority == 1` 的条件做实验；
2. 将 `PENDING` 替换为真实实验结果；
3. 保存为新的 round CSV；
4. 再次运行 `EDBOplus().run(...)` 获取下一批推荐。

若目标只关注产率，可以把 `OBJECTIVES` 改为 `["yield_percent"]`、`OBJECTIVE_MODE` 改为 `["max"]`。若要加入成本、安全性、电压或反应时间，也可以增加目标列，并相应设置 `max` 或 `min`。